# Lab-4: Data Definition & Schema Management

---
When you write your own data types, constraints, and directory structure, you stop thinking of Spark as a "black box" that just translates files.
Without knowledge of Spark's DDL, you'll be relying on Spark to "guess" what it will read from CSV or JSON. 

A professional developer **should dictate the data types** (Integer or Long, Decimal or Float) to avoid rounding errors and memory issues.

## Establishe connection to Spark

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Lab-4-DataDefinition") \
    .config("spark.ui.port", "4040") \
    .config("spark.ui.enabled", "true") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "lab_4/lakehouse") \
    .config("spark.databricks.delta.schema.defaultColumns.enabled", "true")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark version: {spark.version} with Delta support Lab-4")

Spark version: 3.5.0 with Delta support Lab-4


##  1. Create "Banking Schema" using SQL

In [2]:
from pathlib import Path
location_pth="lab_4/lakehouse/bank"
print(f"Create location path: {location_pth}")
folder_path = Path(location_pth)
folder_path.mkdir(parents=True, exist_ok=True)


Create location path: lab_4/lakehouse/bank


In [3]:
print("1. Create namespace")
spark.sql("""CREATE DATABASE IF NOT EXISTS bank_warehouse""")
spark.sql("USE bank_warehouse")

print("2. Create the main table with data types and comments")
spark.sql( 
            """
            CREATE TABLE clients (
                client_id    BIGINT       NOT NULL COMMENT 'Primary Key',
                first_name   STRING       NOT NULL,
                last_name    STRING       NOT NULL,
                email        STRING,
                account_type STRING       COMMENT 'Reference to account_types',
                balance      DECIMAL(18,2) ,
                is_active    BOOLEAN      ,
                created_at   TIMESTAMP,
                updated_at   TIMESTAMP
            )
            USING DELTA
            LOCATION 'bank'
            """
         )       

print("Finish")
          

1. Create namespace
2. Create the main table with data types and comments
Finish


## Create dictionary and put data into

In [4]:
print("Create dictionary [account_types]")

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS account_types (
        code STRING NOT NULL,
        description STRING
    ) USING DELTA
    """
)

print("Insert data into dictionary (DML)")
spark.sql(
    """
    INSERT INTO account_types VALUES 
    ('SAVINGS', 'Saving account'),
    ('CURRENT', 'current account'),
    ('CREDIT', 'loan overdraft')
    """
)
print("finish")

Create dictionary [account_types]
Insert data into dictionary (DML)
finish


## Constraints

- More then one constrain per column:

```py
ALTER TABLE clients ADD CONSTRAINT balance_min CHECK (balance >= -1000);
ALTER TABLE clients ADD CONSTRAINT balance_max CHECK (balance <= 1000000);
```
- Constraint for 2 columns (Multi-column):

```py
ALTER TABLE clients ADD CONSTRAINT check_dates CHECK (updated_at >= created_at);
```

In [5]:
print("Add CHECK constraint ( like in Oracle)")
spark.sql("""ALTER TABLE clients ADD CONSTRAINT check_balance CHECK (balance >= -1000.00)""")
print("Add CHECK constraint - OK!!!!")


Add CHECK constraint ( like in Oracle)
Add CHECK constraint - OK!!!!


In [12]:
print( "Try to break constraine. (Expect erroo)")
try:
    spark.sql("""INSERT INTO clients (client_id, first_name, last_name , balance) VALUES (999, 'Nicola','Tesla',-5000.00)""")
except Exception as e:
    print(f"Констрейнт спрацював!")
    #print(f"Exception details {e}")

Try to break constraine. (Expect erroo)
Констрейнт спрацював!


In [19]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumColumns", 100)
spark.sql("DESCRIBE EXTENDED bank_warehouse.clients")
##.show(truncate=False)

col_name,data_type,comment
client_id,bigint,Primary Key
first_name,string,NULL
last_name,string,NULL
email,string,NULL
account_type,string,Reference to acco...
balance,"decimal(18,2)",NULL
is_active,boolean,NULL
created_at,timestamp,NULL
updated_at,timestamp,NULL
,,


In [22]:
spark.sql("SHOW TBLPROPERTIES bank_warehouse.clients")
##.show(truncate=False)

key,value
delta.constraints...,balance >= - 1000.00
delta.minReaderVe...,1
delta.minWriterVe...,3


## 2. Data clarning (Silver Layer)

In [ ]:
spark.sql("""DROP TABLE clients""")

## Schema Evolution

In [23]:
# Add column using SQL
spark.sql("ALTER TABLE clients ADD COLUMNS (risk_level STRING AFTER account_type)")

# Alternative using Python API
"""
df_with_new_col.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("clients")
"""    

'\ndf_with_new_col.write.format("delta")     .mode("append")     .option("mergeSchema", "true")     .saveAsTable("clients")\n'

In [29]:
p_df=spark.sql("DESCRIBE EXTENDED bank_warehouse.clients").toPandas()
p_df.head(40)

,col_name,data_type,comment
0,client_id,bigint,Primary Key
1,first_name,string,None
2,last_name,string,None
3,email,string,None
4,account_type,string,Reference to account_types
5,risk_level,string,None
6,balance,"decimal(18,2)",None
7,is_active,boolean,None
8,created_at,timestamp,None
9,updated_at,timestamp,None
